In [ ]:
import pandas as pd
import numpy as np

**First of all Load the dataset**

In [ ]:
df = pd.read_csv('/content/Dataset .csv',encoding='utf-8-sig')

In [ ]:
df.head()

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


Based on dataset , classification of dataset is as follow....

---NUMERICAL COLUMN---
*   Restaurant ID
*   Country Code
*   Longitude
*   Latitude
*   Average Cost for two
*   Price range
*   Price range
*   Votes

---CATEGORICAL COLUMN---
*   Restaurant Name
*   City
*   Address
*   Locality
*   Locality Verbose
*   Cuisines
*   Currency
*   Has Table booking
*   Has Online delivery
*   Is delivering now
*   Switch to order menu
*   Rating color
*   Rating text

















In [ ]:
df.dtypes

,0
Restaurant ID,int64
Restaurant Name,object
Country Code,int64
City,object
Address,object
Locality,object
Locality Verbose,object
Longitude,float64
Latitude,float64
Cuisines,object


In [ ]:
df.describe()

,Restaurant ID,Country Code,Longitude,Latitude,Average Cost for two,Price range,Aggregate rating,Votes
count,9.551000e+03,9551.000000,9551.000000,9551.000000,9551.000000,9551.000000,9551.000000,9551.000000
mean,9.051128e+06,18.365616,64.126574,25.854381,1199.210763,1.804837,2.666370,156.909748
std,8.791521e+06,56.750546,41.467058,11.007935,16121.183073,0.905609,1.516378,430.169145
min,5.300000e+01,1.000000,-157.948486,-41.330428,0.000000,1.000000,0.000000,0.000000
25%,3.019625e+05,1.000000,77.081343,28.478713,250.000000,1.000000,2.500000,5.000000
50%,6.004089e+06,1.000000,77.191964,28.570469,400.000000,2.000000,3.200000,31.000000
75%,1.835229e+07,1.000000,77.282006,28.642758,700.000000,2.000000,3.700000,131.000000
max,1.850065e+07,216.000000,174.832089,55.976980,800000.000000,4.000000,4.900000,10934.000000


# **Basic Inspection**

In [ ]:
print("Dataset Shape:", df.shape)

Dataset Shape: (9551, 21)


In [ ]:
print("\nMissing Values Per Column:\n", df.isnull().sum())


Missing Values Per Column:
 Restaurant ID           0
Restaurant Name         0
Country Code            0
City                    0
Address                 0
Locality                0
Locality Verbose        0
Longitude               0
Latitude                0
Cuisines                9
Average Cost for two    0
Currency                0
Has Table booking       0
Has Online delivery     0
Is delivering now       0
Switch to order menu    0
Price range             0
Aggregate rating        0
Rating color            0
Rating text             0
Votes                   0
dtype: int64


There is missing values in the Column having name - **"Cuisines"** missing values = **9**

## For 'Cuisines', which is categorical, we'll fill with 'Unknown'

In [ ]:
df['Cuisines'] = df['Cuisines'].fillna('Unknown')

**DATA TYPE CONVERSIONS**

In [ ]:
df['Aggregate rating'] = pd.to_numeric(df['Aggregate rating'], errors='coerce')

In [ ]:
df['Votes'] = pd.to_numeric(df['Votes'], errors='coerce')

# **REMOVING DUPLICATES**

In [ ]:
initial_rows = len(df)
df.drop_duplicates(inplace=True)
print(f"\nRemoved {initial_rows - len(df)} duplicate rows.")


Removed 0 duplicate rows.


# **FEATURE REFINEMENT**

### The 'Cuisines' column contains multiple values.
### We can create a 'Cuisine_Count' to see variety offered by restaurants.

In [ ]:
df['Cuisine_Count'] = df['Cuisines'].apply(lambda x: len(x.split(',')))

In [ ]:
df['Cuisine_Count']

,Cuisine_Count
0,3
1,1
2,4
3,2
4,2
...,...
9546,1
9547,3
9548,2
9549,1


# **OUTLIER HANDLING**

### **# Some 'Average Cost for two' values might be extreme due to currency differences.**
### We'll use the IQR method to cap extreme outliers if needed for specific analysis.

In [ ]:
Q1 = df['Average Cost for two'].quantile(0.25)

In [ ]:
Q3 = df['Average Cost for two'].quantile(0.75)

In [ ]:
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

In [ ]:
outliers = df[(df['Average Cost for two'] < lower_bound) | (df['Average Cost for two'] > upper_bound)]
print(f"Potential outliers in 'Average Cost for two': {len(outliers)}")

Potential outliers in 'Average Cost for two': 853


# **STANDARDIZING TEXT**

In [ ]:
# Clean trailing spaces in string columns
text_cols = df.select_dtypes(include=['object']).columns
for col in text_cols:
    df[col] = df[col].str.strip()

In [ ]:
print("Fixed City Names Sample:")
print(df['City'].unique()[-10:])

Fixed City Names Sample:
['Doha' 'Cape Town' 'Inner City' 'Johannesburg' 'Pretoria' 'Randburg'
 'Sandton' 'Colombo' 'Ankara' '��stanbul']


Remove any remaining non-ascii characters from the whole dataframe

In [ ]:
def clean_text(text):
    if isinstance(text, str):
        # This removes symbols that couldn't be decoded
        return text.encode('ascii', 'ignore').decode('ascii')
    return text

In [ ]:
df = df.applymap(clean_text)

/tmp/ipykernel_895/2197956661.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_text)


In [ ]:
print("\n--- Cleaning Complete: Special characters handled ---")
print(df[['Restaurant Name', 'City', 'Address']].head())


--- Cleaning Complete: Special characters handled ---
          Restaurant Name              City  \
0        Le Petit Souffle       Makati City   
1        Izakaya Kikufuji       Makati City   
2  Heat - Edsa Shangri-La  Mandaluyong City   
3                    Ooma  Mandaluyong City   
4             Sambo Kojin  Mandaluyong City   

                                             Address  
0  Third Floor, Century City Mall, Kalayaan Avenu...  
1  Little Tokyo, 2277 Chino Roces Avenue, Legaspi...  
2  Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...  
3  Third Floor, Mega Fashion Hall, SM Megamall, O...  
4  Third Floor, Mega Atrium, SM Megamall, Ortigas...  


In [ ]:
print("\nCleaned Data Info:")
print(df.info())


Cleaned Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Restaurant ID         9551 non-null   int64  
 1   Restaurant Name       9551 non-null   object 
 2   Country Code          9551 non-null   int64  
 3   City                  9551 non-null   object 
 4   Address               9551 non-null   object 
 5   Locality              9551 non-null   object 
 6   Locality Verbose      9551 non-null   object 
 7   Longitude             9551 non-null   float64
 8   Latitude              9551 non-null   float64
 9   Cuisines              9551 non-null   object 
 10  Average Cost for two  9551 non-null   int64  
 11  Currency              9551 non-null   object 
 12  Has Table booking     9551 non-null   object 
 13  Has Online delivery   9551 non-null   object 
 14  Is delivering now     9551 non-null   object 
 15  S

In [ ]:
df.to_csv('Cleaned_Restaurant_Dataset.csv', index=False)

In [ ]:
df = pd.read_csv('/content/Cleaned_Restaurant_Dataset.csv')

In [ ]:
df['Cuisines'].tail(8)

,Cuisines
9543,Cafe
9544,"Desserts, B_rek"
9545,"Burger, Izgara"
9546,Turkish
9547,"World Cuisine, Patisserie, Cafe"
9548,"Italian, World Cuisine"
9549,Restaurant Cafe
9550,Cafe


In [ ]:
print("\nMissing Values Per Column:\n", df.isnull().sum())


Missing Values Per Column:
 Restaurant ID           0
Restaurant Name         0
Country Code            0
City                    0
Address                 0
Locality                0
Locality Verbose        0
Longitude               0
Latitude                0
Cuisines                0
Average Cost for two    0
Currency                0
Has Table booking       0
Has Online delivery     0
Is delivering now       0
Switch to order menu    0
Price range             0
Aggregate rating        0
Rating color            0
Rating text             0
Votes                   0
Cuisine_Count           0
dtype: int64
